This notebook is for modeling and evaluating the SemEval dataset

In [18]:
import pandas as pd
import numpy as np
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification
from datasets import Dataset
import evaluate

ModuleNotFoundError: No module named 'evaluate'

In [12]:
#Set up paths
BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data" / "processed"
MODEL_DIR = BASE_DIR / "models" / "semeval_bert_scanner"

In [13]:
#Check for GPU support
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using Apple Metal (MPS) for acceleration")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using NVIDIA GPU")
else:
    device = torch.device("cpu")
    print("Using CPU. Training might be slow.")

Using Apple Metal (MPS) for acceleration


In [16]:
#Load span identification data
df_si = pd.read_csv(DATA_DIR / "semeval_si_cleaned.csv")
df_si.head()

,article_id,sentence_text,label
0,111111111,Next plague outbreak in Madagascar could be 's...,1
1,111111111,"""The next transmission could be more pronounce...",1
2,111111111,"An outbreak of both bubonic plague, which is s...",0
3,111111111,Madagascar has suffered bubonic plague outbrea...,0
4,111111111,The disease tends to make a comeback each hot ...,0


In [17]:
#Load technique classification data
df_tc = pd.read_csv(DATA_DIR / "semeval_tc_cleaned.csv")
df_tc.head()

,article_id,technique,start_char,end_char,source_file,text_content,technique_list,span_text,sentiment,punct_count,lexical_diversity
0,758756657,Repetition,5024,5036,article758756657.task2-TC.labels,Islamizing the Schools: The Case of West Virgi...,Repetition,Islamization,0.000000,0,1.000000
1,758756657,Repetition,5302,5314,article758756657.task2-TC.labels,Islamizing the Schools: The Case of West Virgi...,Repetition,Islamization,0.000000,0,1.000000
2,758756657,Loaded_Language,62,69,article758756657.task2-TC.labels,Islamizing the Schools: The Case of West Virgi...,Loaded_Language,outrage,0.000000,0,1.000000
3,758756657,Causal_Oversimplification,606,746,article758756657.task2-TC.labels,Islamizing the Schools: The Case of West Virgi...,Causal_Oversimplification,"In order to convert to Islam, one says the sha...",-0.166667,0,0.785714
4,758756657,Loaded_Language,4352,4361,article758756657.task2-TC.labels,Islamizing the Schools: The Case of West Virgi...,Loaded_Language,egregious,0.000000,0,1.000000
